<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/P11/04_SiRNA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNA de Interferência

Neste último Colab, vamos treinar um modelo para indicar se uma sequência de RNA é uma boa sequência para causar o **silenciamento** de um gene.

O RNA de interferência foi descoberto nos anos 1990 e rendeu um prêmio Nobel em 2006. De forma muito resumida, um fragmento de RNA cuja sequência seja complementar a uma parte de um mRNA pode se ligar ao mRNA. O resultado é o *silenciamento* do mRNA, isto é, ele deixa de ser traduzido em uma proteína. O fragmento de RNA que silencia o gene é chamado de *short interfering RNA* ou siRNA. Este mecanismo permite "*desligar*" a transcrição de um gene e é muito empregado para a pesquisa sobre a importância do gene em determinado contexto.

Uma vez que as sequências de RNA não são aleatórias, e que o RNA possui uma estrutura complexa, diferentes sequências de RNA apresentam eficiências diferentes como siRNA e seria interessante ter uma ferramenta que selecionasse as melhores sequências para um siRNA.

# Deep Learning

Uma biblioteca de 2187 moléculas de siRNA foram testadas experimentalmente, cada uma possuindo 21 bases. Cada uma recebeu um *score* numérico (um valor entre 0 e 1) indicando o quão efetivo foi o silenciamento do gene alvo.

Vamos gerar um modelo para, a partir destes dados, estimar a qualidade da sequência como uma sequência para siRNA.

## Bibliotecas

Como na prática anterior, esta versão lê os dados direto dos arquivos `.joblib` e usa o Keras do
próprio Colab — sem DeepChem.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from scipy.stats import pearsonr
from sklearn.metrics import mean_squared_error

print("TensorFlow:", tf.__version__)

## Baixando os dados e carregando no ambiente

In [ ]:
import os
if not os.path.isdir('DeepLearningLifeSciences'):
    !git clone --depth 1 -q https://github.com/deepchem/DeepLearningLifeSciences.git
%cd -q DeepLearningLifeSciences/Chapter06
!ls

In [ ]:
import joblib

def carregar(pasta):
    X = joblib.load(f"{pasta}/shard-0-X.joblib")
    y = joblib.load(f"{pasta}/shard-0-y.joblib")
    return np.asarray(X, "float32"), np.asarray(y, "float32").ravel()

X_tr, y_tr = carregar('train_siRNA')
X_va, y_va = carregar('valid_siRNA')
print("treino:   ", X_tr.shape, y_tr.shape)
print("validação:", X_va.shape, y_va.shape)
print("escore de silenciamento: min %.3f, média %.3f, máx %.3f" % (y_tr.min(), y_tr.mean(), y_tr.max()))

In [ ]:
# Distribuição dos escores - o que estamos tentando prever?
plt.figure(figsize=(5.5, 3.5))
plt.hist(y_tr, bins=30, color='#4C78A8')
plt.xlabel('escore de silenciamento'); plt.ylabel('nº de siRNAs')
plt.title('Alvo da regressão (treino)'); plt.tight_layout(); plt.show()

## Uma referência antes do modelo complicado

Antes de treinar uma rede, vale saber o que um modelo **linear** consegue fazer com as mesmas
características. Se a rede não superar essa linha de base, ela não está agregando nada.

Este é um hábito que vale levar para a vida: *toda* comparação de modelo precisa de uma referência.

In [ ]:
from sklearn.linear_model import RidgeCV
base = RidgeCV(alphas=np.logspace(-2, 3, 20)).fit(X_tr.reshape(len(X_tr), -1), y_tr)
pred_base = base.predict(X_va.reshape(len(X_va), -1))
r_base = pearsonr(y_va, pred_base)[0]
print("Regressão linear (ridge) - Pearson r na validação: %.3f" % r_base)

## Construção do modelo

Mais uma vez uma CNN 1D. A diferença importante é que agora o modelo é de **regressão**: queremos
prever um número contínuo (o escore de silenciamento), não uma classe. Por isso a função de perda é
o **erro quadrático médio**, que minimiza a diferença quadrática entre predito e observado.

**Q1: A camada de saída usa ativação sigmoide. Por que isso é adequado aqui, e em que situação
seria um erro?**

In [ ]:
modelo = keras.Sequential([
    keras.Input(shape=(21, 4)),
    layers.Conv1D(10, kernel_size=10, activation='relu', padding='same'),
    layers.Dropout(0.3),
    layers.Conv1D(10, kernel_size=10, activation='relu', padding='same'),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(1, activation='sigmoid'),
])
modelo.compile(optimizer='adam', loss='mse', metrics=['mae'])
modelo.summary()

## Treinamento

**Atenção ao tamanho do lote.** O conjunto de treino tem cerca de 1800 sequências. Com
`batch_size=1000`, cada época dá **2** passos de gradiente — 200 épocas somam apenas ~400
atualizações, e o modelo fica subtreinado por construção. Com lotes de 32, a mesma época dá ~57
passos.

**Q2: Rode o treino abaixo, anote o `r` final; depois volte, troque `batch_size` para 1000, rode de
novo e compare. Quantos passos de gradiente cada configuração executa por época?**

In [ ]:
parada = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10,
                                       restore_best_weights=True)

historico = modelo.fit(X_tr, y_tr,
                       validation_data=(X_va, y_va),
                       epochs=200, batch_size=32,
                       callbacks=[parada], verbose=0)
print("épocas efetivamente rodadas:", len(historico.history['loss']))

## Métrica

Como a predição é uma regressão, usamos a correlação de **Pearson** entre valores reais e preditos:
um número entre -1 (anticorrelação) e 1 (correlação perfeita), com 0 indicando ausência de relação.
Junto, o erro quadrático médio, que está na mesma unidade do alvo.

In [ ]:
p_tr = modelo.predict(X_tr, verbose=0).ravel()
p_va = modelo.predict(X_va, verbose=0).ravel()

print("Pearson r [treino]:    %.3f" % pearsonr(y_tr, p_tr)[0])
print("Pearson r [validação]: %.3f" % pearsonr(y_va, p_va)[0])
print("RMSE      [validação]: %.4f" % mean_squared_error(y_va, p_va) ** 0.5)
print("\nreferência linear (ridge): r = %.3f" % r_base)

## Análise gráfica

In [ ]:
h = historico.history
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(h['loss'], label='treino'); a1.plot(h['val_loss'], label='validação')
a1.set_xlabel('época'); a1.set_ylabel('erro quadrático médio'); a1.legend(); a1.set_title('Treinamento')

a2.scatter(y_va, p_va, s=14, alpha=0.6)
lim = [min(y_va.min(), p_va.min()), max(y_va.max(), p_va.max())]
a2.plot(lim, lim, '--', color='gray')
a2.set_xlabel('escore observado'); a2.set_ylabel('escore predito')
a2.set_title('Validação: predito x observado (r = %.3f)' % pearsonr(y_va, p_va)[0])
plt.tight_layout(); plt.show()

**Q3: A partir das curvas, o modelo está bom? O treinamento precisa de mais rodadas? Justifique.**

**Q4: O gráfico de dispersão mostra uma nuvem achatada em relação à diagonal: o modelo acerta a
tendência mas comprime a faixa de valores. O que isso diz sobre o que ele aprendeu?**

**Q5: A rede superou a referência linear? Se o ganho for pequeno, o que isso sugere sobre a
quantidade de dados (2187 sequências) em relação ao número de parâmetros do modelo?**

**Q6: Aqui só existem treino e validação. Se você fosse publicar este preditor, o que faltaria?**